#### Ispezione cammini BGP

Dovrebbe essere del tipo: routeviews/routeviews|5 1239|6113|8063|3464 199.88.21.0/24 i 144.228.240.93

In [2]:
import bz2

with bz2.open("/code/ADSproject/data/20110501.all-paths.bz2", "rt") as f:
    for i, riga in enumerate(f):
        print(riga)
        if i > 5:  # guarda solo le prime righe
            break

routeviews/isc|5 4436|6762|21826 200.82.128.0/24 i 198.32.176.13

routeviews/isc|5 6939|15290|2671|2669 198.103.53.0/24 i 198.32.176.20

routeviews/isc|5 4826|6939|3549|7922|20214 70.89.88.0/24 i 198.32.176.134

routeviews/isc|5 14361|15290|23373 216.9.51.0/24 i 198.32.176.10

routeviews/isc|5 14361|4766|1237 203.254.160.0/21 i 198.32.176.10

routeviews/isc|5 4589|3356|12956|3816|8163 190.182.7.0/24 i 198.32.176.74

routeviews/isc|5 7575|6939|22637 67.208.228.0/24 i 198.32.176.177



#### Ispezione nodi

Dovrebbe essere del tipo: AS1|AS2|relationship (relationship non ci interessa)

In [3]:
import bz2

with bz2.open("/code/ADSproject/data/20110501.as-rel.txt.bz2", "rt") as f:
    for i, riga in enumerate(f):
        print(riga)
        if i > 5:  # guarda solo le prime righe
            break

# source:topology|BGP|20110501|ripe|rrc00

# source:topology|BGP|20110502|ripe|rrc00

# source:topology|BGP|20110503|ripe|rrc00

# source:topology|BGP|20110504|ripe|rrc00

# source:topology|BGP|20110505|ripe|rrc00

# source:topology|BGP|20110501|ripe|rrc01

# source:topology|BGP|20110502|ripe|rrc01



In [4]:
import bz2

with bz2.open("/code/ADSproject/data/20110501.as-rel.txt.bz2", "rt") as f:
    for i, riga in enumerate(f):
        if riga.startswith("#"):
            continue  # salta i commenti
        print(riga.strip())
        break  # guarda solo la prima riga utile

1|2|-1


#### Prova estrazione dati (parsing dei cammini dal file all paths)

In [5]:
def leggi_cammini(filepath, max_righe=None):
    cammini = []
    contatore = 0
    with bz2.open(filepath, "rt") as f:
        for riga in f:
            if max_righe and contatore >= max_righe:
                break
            if riga.startswith("#"):
                continue
            parti = riga.strip().split()
            print(parti)
            cammino = []
            for p in parti[1:]:
                if "/" in p or "." in p or ":" in p:  
                    break
                nodi = p.split("|")
                print(nodi)
                cammino.extend(nodi)
            if len(cammino) > 1:  ## aggiungi solo cammini con almeno due nodi 
                cammini.append(cammino)
                contatore += 1
    return cammini

In [6]:
# test
cammini = leggi_cammini("/code/ADSproject/data/20110501.all-paths.bz2", max_righe=1000)  # leggi solo le prime 10 righe per test
print(cammini[:5])  # stampa i primi 5 cammini


['routeviews/isc|5', '4436|6762|21826', '200.82.128.0/24', 'i', '198.32.176.13']
['4436', '6762', '21826']
['routeviews/isc|5', '6939|15290|2671|2669', '198.103.53.0/24', 'i', '198.32.176.20']
['6939', '15290', '2671', '2669']
['routeviews/isc|5', '4826|6939|3549|7922|20214', '70.89.88.0/24', 'i', '198.32.176.134']
['4826', '6939', '3549', '7922', '20214']
['routeviews/isc|5', '14361|15290|23373', '216.9.51.0/24', 'i', '198.32.176.10']
['14361', '15290', '23373']
['routeviews/isc|5', '14361|4766|1237', '203.254.160.0/21', 'i', '198.32.176.10']
['14361', '4766', '1237']
['routeviews/isc|5', '4589|3356|12956|3816|8163', '190.182.7.0/24', 'i', '198.32.176.74']
['4589', '3356', '12956', '3816', '8163']
['routeviews/isc|5', '7575|6939|22637', '67.208.228.0/24', 'i', '198.32.176.177']
['7575', '6939', '22637']
['routeviews/isc|5', '4436|701|702', '213.116.184.0/23', 'i', '198.32.176.13']
['4436', '701', '702']
['routeviews/isc|5', '4436|2914|7018|12217', '153.2.77.0/24', 'i', '198.32.176.13'

#### Prova costruzione frequenze a partire dai cammini estratti 
(lista di liste del tipo [['4436', '6762', '21826'], ['6939', '15290', '2671', '2669']])

In [7]:
from collections import defaultdict

frequenze = defaultdict(int)

for cammino in cammini:
        for i in range(len(cammino) - 1):
            u = cammino[i]
            v = cammino[i + 1]
            if u == v:  # ignora self-loop
                continue
            # stampa l'arco
            print(f"Arco: {u} -> {v}")
            u_int, v_int = int(u), int(v)
            arco = (min(u_int, v_int), max(u_int, v_int)) ## per non avere archi duplicati del tipo (1,2) e (2,1)
            frequenze[arco] = frequenze.get(arco, 0) + 1
            print(f"Frequenza arco {u} -> {v}: {frequenze[arco]}")

Arco: 4436 -> 6762
Frequenza arco 4436 -> 6762: 1
Arco: 6762 -> 21826
Frequenza arco 6762 -> 21826: 1
Arco: 6939 -> 15290
Frequenza arco 6939 -> 15290: 1
Arco: 15290 -> 2671
Frequenza arco 15290 -> 2671: 1
Arco: 2671 -> 2669
Frequenza arco 2671 -> 2669: 1
Arco: 4826 -> 6939
Frequenza arco 4826 -> 6939: 1
Arco: 6939 -> 3549
Frequenza arco 6939 -> 3549: 1
Arco: 3549 -> 7922
Frequenza arco 3549 -> 7922: 1
Arco: 7922 -> 20214
Frequenza arco 7922 -> 20214: 1
Arco: 14361 -> 15290
Frequenza arco 14361 -> 15290: 1
Arco: 15290 -> 23373
Frequenza arco 15290 -> 23373: 1
Arco: 14361 -> 4766
Frequenza arco 14361 -> 4766: 1
Arco: 4766 -> 1237
Frequenza arco 4766 -> 1237: 1
Arco: 4589 -> 3356
Frequenza arco 4589 -> 3356: 1
Arco: 3356 -> 12956
Frequenza arco 3356 -> 12956: 1
Arco: 12956 -> 3816
Frequenza arco 12956 -> 3816: 1
Arco: 3816 -> 8163
Frequenza arco 3816 -> 8163: 1
Arco: 7575 -> 6939
Frequenza arco 7575 -> 6939: 1
Arco: 6939 -> 22637
Frequenza arco 6939 -> 22637: 1
Arco: 4436 -> 701
Frequenz

In [8]:
frequenze

defaultdict(int,
            {(4436, 6762): 5,
             (6762, 21826): 1,
             (6939, 15290): 4,
             (2671, 15290): 2,
             (2669, 2671): 2,
             (4826, 6939): 80,
             (3549, 6939): 10,
             (3549, 7922): 1,
             (7922, 20214): 1,
             (14361, 15290): 1,
             (15290, 23373): 1,
             (4766, 14361): 2,
             (1237, 4766): 2,
             (3356, 4589): 51,
             (3356, 12956): 5,
             (3816, 12956): 5,
             (3816, 8163): 3,
             (6939, 7575): 4,
             (6939, 22637): 1,
             (701, 4436): 11,
             (701, 702): 5,
             (2914, 4436): 36,
             (2914, 7018): 22,
             (7018, 12217): 1,
             (1273, 7575): 5,
             (1273, 20485): 3,
             (20485, 21127): 2,
             (3491, 19151): 3,
             (3549, 4436): 15,
             (3549, 5396): 1,
             (5396, 51940): 1,
             (19151, 34224): 1,

In [9]:
for arco, freq in frequenze.items():
        print(f"Arco: {arco}, Frequenza: {freq}")

Arco: (4436, 6762), Frequenza: 5
Arco: (6762, 21826), Frequenza: 1
Arco: (6939, 15290), Frequenza: 4
Arco: (2671, 15290), Frequenza: 2
Arco: (2669, 2671), Frequenza: 2
Arco: (4826, 6939), Frequenza: 80
Arco: (3549, 6939), Frequenza: 10
Arco: (3549, 7922), Frequenza: 1
Arco: (7922, 20214), Frequenza: 1
Arco: (14361, 15290), Frequenza: 1
Arco: (15290, 23373), Frequenza: 1
Arco: (4766, 14361), Frequenza: 2
Arco: (1237, 4766), Frequenza: 2
Arco: (3356, 4589), Frequenza: 51
Arco: (3356, 12956), Frequenza: 5
Arco: (3816, 12956), Frequenza: 5
Arco: (3816, 8163), Frequenza: 3
Arco: (6939, 7575), Frequenza: 4
Arco: (6939, 22637), Frequenza: 1
Arco: (701, 4436), Frequenza: 11
Arco: (701, 702), Frequenza: 5
Arco: (2914, 4436), Frequenza: 36
Arco: (2914, 7018), Frequenza: 22
Arco: (7018, 12217), Frequenza: 1
Arco: (1273, 7575), Frequenza: 5
Arco: (1273, 20485), Frequenza: 3
Arco: (20485, 21127), Frequenza: 2
Arco: (3491, 19151), Frequenza: 3
Arco: (3549, 4436), Frequenza: 15
Arco: (3549, 5396), Fr

In [10]:
frequenze

defaultdict(int,
            {(4436, 6762): 5,
             (6762, 21826): 1,
             (6939, 15290): 4,
             (2671, 15290): 2,
             (2669, 2671): 2,
             (4826, 6939): 80,
             (3549, 6939): 10,
             (3549, 7922): 1,
             (7922, 20214): 1,
             (14361, 15290): 1,
             (15290, 23373): 1,
             (4766, 14361): 2,
             (1237, 4766): 2,
             (3356, 4589): 51,
             (3356, 12956): 5,
             (3816, 12956): 5,
             (3816, 8163): 3,
             (6939, 7575): 4,
             (6939, 22637): 1,
             (701, 4436): 11,
             (701, 702): 5,
             (2914, 4436): 36,
             (2914, 7018): 22,
             (7018, 12217): 1,
             (1273, 7575): 5,
             (1273, 20485): 3,
             (20485, 21127): 2,
             (3491, 19151): 3,
             (3549, 4436): 15,
             (3549, 5396): 1,
             (5396, 51940): 1,
             (19151, 34224): 1,

In [11]:
dizio = defaultdict(dict)

In [12]:
dizio[0]

{}

In [13]:
dizio

defaultdict(dict, {0: {}})

In [14]:
dizio[0][5] = 3

In [15]:
dizio

defaultdict(dict, {0: {5: 3}})

#### Prova costruzione grafo

In [16]:
from collections import defaultdict

frequenze = defaultdict(int)

for cammino in cammini:
        for i in range(len(cammino) - 1):
            u = cammino[i]
            v = cammino[i + 1]
            if u == v:  # ignora self-loop
                continue
            # stampa l'arco
            print(f"Arco: {u} -> {v}")
            u_int, v_int = int(u), int(v)
            arco = (min(u_int, v_int), max(u_int, v_int))
            frequenze[arco] = frequenze.get(arco, 0) + 1
            print(f"Frequenza arco {u} -> {v}: {frequenze[arco]}")


grafo = defaultdict(dict)  # grafo[u][v] = frequenza

for arco, freq in frequenze.items():
    u, v = arco
    print(u, v, freq)
    grafo[u][v] = freq
    grafo[v][u] = freq  # grafo non orientato

print(grafo)

Arco: 4436 -> 6762
Frequenza arco 4436 -> 6762: 1
Arco: 6762 -> 21826
Frequenza arco 6762 -> 21826: 1
Arco: 6939 -> 15290
Frequenza arco 6939 -> 15290: 1
Arco: 15290 -> 2671
Frequenza arco 15290 -> 2671: 1
Arco: 2671 -> 2669
Frequenza arco 2671 -> 2669: 1
Arco: 4826 -> 6939
Frequenza arco 4826 -> 6939: 1
Arco: 6939 -> 3549
Frequenza arco 6939 -> 3549: 1
Arco: 3549 -> 7922
Frequenza arco 3549 -> 7922: 1
Arco: 7922 -> 20214
Frequenza arco 7922 -> 20214: 1
Arco: 14361 -> 15290
Frequenza arco 14361 -> 15290: 1
Arco: 15290 -> 23373
Frequenza arco 15290 -> 23373: 1
Arco: 14361 -> 4766
Frequenza arco 14361 -> 4766: 1
Arco: 4766 -> 1237
Frequenza arco 4766 -> 1237: 1
Arco: 4589 -> 3356
Frequenza arco 4589 -> 3356: 1
Arco: 3356 -> 12956
Frequenza arco 3356 -> 12956: 1
Arco: 12956 -> 3816
Frequenza arco 12956 -> 3816: 1
Arco: 3816 -> 8163
Frequenza arco 3816 -> 8163: 1
Arco: 7575 -> 6939
Frequenza arco 7575 -> 6939: 1
Arco: 6939 -> 22637
Frequenza arco 6939 -> 22637: 1
Arco: 4436 -> 701
Frequenz

In [17]:
dizio

defaultdict(dict, {0: {5: 3}})

In [18]:
dizio[6] = {}

In [19]:
dizio[7][6] = 4

In [20]:
dizio

defaultdict(dict, {0: {5: 3}, 6: {}, 7: {6: 4}})

In [21]:
def remove_node(self, node):
        if node not in self:
            raise ValueError(
                f"Node {node} does not exist in the graph."
            )

        for neighbors in self.values():
            neighbors.pop(node, None)

        del self[node]

In [22]:
remove_node(dizio, 6)

In [23]:
dizio

defaultdict(dict, {0: {5: 3}, 7: {}})

In [24]:
class Graph:

    def __init__(self, directed=False):

        self.adjacency_list = {}  # il grafo sarà un dizionario di dizionari: {nodo: {vicino: peso}} (non usiamo defaultdict per avere più controllo)
        self.directed = directed  # default è False, quindi il grafo è non orientato

## Converte l'identificatore del nodo AS che è una stringa in un intero.

    def _convert_node(self, node):

        try:
            return int(node)
        except (TypeError, ValueError):
            raise ValueError(f"Node {node} is not a valid integer identifier.")

## Definisce come stampare il grafo in modo leggibile

    def __repr__(self):
        graph_str = ""
        for node, neighbors in self.adjacency_list.items():
            graph_str += f" Node {node}: Neighbors and weights {neighbors} \n"
        return graph_str

## Aggiunge un nodo al grafo. Se il nodo esiste già, solleva un'eccezione.

    def add_node(self, node):
        node = self._convert_node(node)

        if node not in self.adjacency_list:
            self.adjacency_list[node] = {}  # aggiunge il nodo con un dizionario (lista di adiacenza e pesi) vuoto
        else:
            raise ValueError(f"Node {node} already exists in the graph.")

## Rimuove un nodo dal grafo. Se il nodo non esiste, solleva un'eccezione.

    def remove_node(self, node):
        node = self._convert_node(node)

        if node not in self.adjacency_list:
            raise ValueError(f"Node {node} does not exist in the graph.")

        for neighbors in self.adjacency_list.values(): ## rimuove il nodo da tutte le liste di adiacenza dei vicini
            neighbors.pop(node, None)

        del self.adjacency_list[node]

## Aggiunge un arco al grafo. Questo arco può essere arbitrario e non proveniente dai cammini BGP (il suo peso sarà None o specificato arbitrariamente dall'utente)

    def add_edge(self, from_node, to_node, weight=None):
        

        #from_node = self._convert_node(from_node) ##ridondanti, lo fa già add_node
        #to_node = self._convert_node(to_node)

        if from_node == to_node:  # elimina i self-loop
            return

        if from_node not in self.adjacency_list:
            self.add_node(from_node)

        if to_node not in self.adjacency_list:
            self.add_node(to_node)

      
        self.adjacency_list[from_node][to_node] = weight

        if not self.directed: ## aggiunge arco in entrambe le direzioni se è undirected
                self.adjacency_list[to_node][from_node] = weight
    
## Rimuove un arco dal grafo. Se i nodi A e B non esistono o se l'arco stesso non esiste (magari i nodi sì ma non sono collegati) lancia un errore.

    def remove_edge(self, from_node, to_node):

        from_node = self._convert_node(from_node)
        to_node = self._convert_node(to_node)

        if from_node not in self.adjacency_list:
            raise ValueError(
                f"Node {from_node} does not exist in the graph."
            )

        if to_node not in self.adjacency_list:
            raise ValueError(
                f"Node {to_node} does not exist in the graph."
            )

        if to_node not in self.adjacency_list[from_node]:
            raise ValueError(
                f"Edge ({from_node}, {to_node}) does not exist in the graph."
            )

        del self.adjacency_list[from_node][to_node]

        if not self.directed:  ## elimina anche l'arco inverso 
            if from_node in self.adjacency_list.get(to_node, {}): ## fa un check se esiste (non dovrebbe servire teoricamente)
                del self.adjacency_list[to_node][from_node]


## Aggiorna la frequenza degli archi

    def update_frequency(self, from_node, to_node):

        from_node = self._convert_node(from_node)
        to_node = self._convert_node(to_node)

        if from_node == to_node:  # elimina i self-loop
            return
        
        ## ho il dubbio che non sia giusto crearli, vediamo

        if from_node not in self.adjacency_list:
            self.add_node(from_node)

        if to_node not in self.adjacency_list:
            self.add_node(to_node)

        #legge la frequenza attuale con .get(...); se l’arco ancora non esiste restituisce 0 come frequenza iniziale e poi aggiunge 1 
        # assegna il nuovo valore a self.adjacency_list[from_node][to_node].

        self.adjacency_list[from_node][to_node] = (                      
            self.adjacency_list[from_node].get(to_node, 0) + 1
        )

        if not self.directed:
            self.adjacency_list[to_node][from_node] = (
                self.adjacency_list[to_node].get(from_node, 0) + 1
            )
    
    def get_neighbors(self, node):
        node = self._convert_node(node)

        if node in self.adjacency_list:
            return self.adjacency_list[node]
        else:
            raise ValueError(f"Node {node} does not exist in the graph.")

    def has_node(self, node):
        node = self._convert_node(node)
        return node in self.adjacency_list

    def has_edge(self, from_node, to_node):
        from_node = self._convert_node(from_node)
        to_node = self._convert_node(to_node)

        if from_node in self.adjacency_list:
            return to_node in self.adjacency_list[from_node]

        return False

    def get_nodes(self):
        return list(self.adjacency_list.keys())

    def get_edges(self):
        edges = []
        seen = set()

        for from_node, neighbors in self.adjacency_list.items():
            for to_node, weight in neighbors.items():

                if not self.directed:
                    arco = (min(from_node, to_node), max(from_node, to_node)) ## prende una unica entry tra (2,3) (3,2)

                    if arco not in seen:
                        seen.add(arco)
                        edges.append((from_node, to_node, weight))

        return edges

    def delete_consecutive_duplicates(self, path): # per esempio path = [10, 10, 20] , conserva solo il primo 10 e 20
        return [
            path[i]
            for i in range(len(path))
            if i == 0 or path[i] != path[i - 1]
        ]

    def add_bgp_path(self, path):
        # converte tutti gli identificatori AS in interi
        path = [self._convert_node(node) for node in path]

        path = self.delete_consecutive_duplicates(path)

        for i in range(len(path) - 1):
            u = path[i]
            v = path[i + 1]

            if u == v:  # elimina i self-loop
                continue

            if not self.has_node(u):
                self.add_node(u)

            if not self.has_node(v):
                self.add_node(v)

            self.update_frequency(u, v)
    

    def largest_connected_component(self):  # Cerchiamo la componente connessa più grande tramite una DFS iterativa.

        visited = set() # nodi già visti

        largest_component = set() # insieme che conterrà i nodi della componente connessa più grande trovata fino a questo momento

        for start_node in self.adjacency_list:

            if start_node in visited:
                continue
                # Saltiamo il resto dell'iterazione e passiamo al nodo successivo se é stato già visto

            component = set() # insieme che conterrà i nodi della componente che stiamo esplorando in questo momento.

            stack = [start_node]
            # Pila utilizzata per eseguire la DFS.
            # Inizialmente contiene solo il nodo di partenza.

            visited.add(start_node)

            while stack:
                # Continuiamo la visita finché ci sono nodi nella pila.

                node = stack.pop()
                # Estraiamo l'ultimo nodo inserito nella pila.
                # Questo comportamento LIFO realizza una DFS.

                component.add(node)
                # Aggiungiamo il nodo alla componente connessa corrente.

                for neighbor in self.adjacency_list[node]:
                    # Scorriamo tutti i vicini del nodo.
                    # Il dizionario interno ha la forma:
                    # {vicino: peso}
                    # Qui vengono considerate solo le chiavi, cioè i vicini.
                    # I pesi non servono per trovare le componenti connesse.

                    if neighbor not in visited:
                        # Consideriamo solo i vicini
                        # che non sono ancora stati visitati.

                        visited.add(neighbor)
                        # Segniamo il vicino come visitato.

                        stack.append(neighbor)
                        # Inseriamo il vicino nella pila,
                        # così verrà esplorato successivamente.

            if len(component) > len(largest_component):
                # Quando la DFS termina, abbiamo trovato
                # un'intera componente connessa.
                # Confrontiamo il suo numero di nodi
                # con quello della componente più grande trovata finora.

                largest_component = component
                # Se la componente corrente è più grande,
                # la salviamo come nuova componente più grande.

        return largest_component
        # Restituiamo l'insieme dei nodi appartenenti
        # alla componente connessa più grande.


    def get_largest_connected_subgraph(self):
        # Troviamo i nodi appartenenti
        # alla componente connessa più grande.

        component_nodes = self.largest_connected_component()

        # Creiamo un nuovo oggetto Graph.
        # Il nuovo grafo mantiene la stessa proprietà del grafo originale:
        # orientato se self.directed è True,
        # non orientato se self.directed è False.

        subgraph = Graph(directed=self.directed)

        for node in component_nodes:
            # Scorriamo tutti i nodi della componente più grande.

            subgraph.add_node(node)
            # Aggiungiamo ogni nodo al nuovo sottografo.
            # In questa fase ogni nodo viene creato
            # con un dizionario dei vicini inizialmente vuoto.

        for from_node in component_nodes:
            # Scorriamo nuovamente tutti i nodi
            # della componente connessa più grande.

            for to_node, weight in self.adjacency_list[from_node].items():
                # Per ogni nodo, scorriamo tutti i suoi vicini
                # e i relativi pesi nel grafo originale.

                if to_node in component_nodes:
                    # Copiamo l'arco soltanto se anche il vicino
                    # appartiene alla componente connessa più grande.

                    subgraph.adjacency_list[from_node][to_node] = weight
                    # Copiamo direttamente l'arco e il suo peso.
                    # Il peso non viene modificato né ricalcolato:
                    # resta uguale a quello presente nel grafo originale.

        return subgraph
        # Restituiamo un nuovo oggetto Graph contenente soltanto
        # la componente connessa più grande.
        # Il grafo originale non viene modificato.



In [25]:
grafo = Graph(directed=False)
for cammino in cammini:
 grafo.add_bgp_path(cammino)

In [26]:
largest_graph = grafo.get_largest_connected_subgraph()

In [27]:
largest_graph

 Node 2048: Neighbors and weights {7911: 1} 
 Node 30721: Neighbors and weights {8764: 1, 16130: 1} 
 Node 10242: Neighbors and weights {174: 1} 
 Node 38913: Neighbors and weights {30890: 1} 
 Node 20485: Neighbors and weights {1273: 3, 21127: 2, 16285: 2, 19151: 2, 6939: 3, 6697: 2, 50544: 1, 4589: 2, 28745: 1, 3491: 1, 31359: 1, 25510: 1, 15835: 1} 
 Node 51206: Neighbors and weights {9121: 1} 
 Node 393223: Neighbors and weights {1273: 1, 11139: 2, 1239: 1} 
 Node 18438: Neighbors and weights {14742: 1} 
 Node 55304: Neighbors and weights {1221: 1} 
 Node 36874: Neighbors and weights {29606: 1} 
 Node 16391: Neighbors and weights {7018: 1} 
 Node 12301: Neighbors and weights {8928: 1, 13124: 1, 6939: 1, 12978: 1} 
 Node 16397: Neighbors and weights {3549: 2, 262744: 1} 
 Node 26640: Neighbors and weights {32105: 1} 
 Node 55314: Neighbors and weights {7602: 1} 
 Node 30739: Neighbors and weights {49808: 1} 
 Node 14361: Neighbors and weights {15290: 1, 4766: 2, 3549: 39, 3491: 3, 7

In [28]:
u = 4826
v = 6939

## per verificare che siano presenti entrambi i collegamenti nelle liste con lo stesso peso

print(grafo.adjacency_list[u][v])  # peso da u a v
print(grafo.adjacency_list[v][u])  # peso da v a u

80
80


In [29]:
import pickle
def load_paths(filepath_bz2=None, filepath_pkl=None, max_paths=None):
        
        if filepath_bz2:
            # legge direttamente dal bz2 fermandosi a max_paths
            cammini = []
            contatore = 0
            with bz2.open(filepath_bz2, "rt") as f:
                for riga in f:
                    if contatore >= max_paths:
                        break
                    if riga.startswith("#"):
                        continue
                    parti = riga.strip().split()
                    cammino = []
                    for p in parti[1:]:
                        if "/" in p or "." in p or ":" in p:
                            break
                        nodi = p.split("|")
                        cammino.extend(nodi)
                    if len(cammino) > 1:
                        cammini.append(cammino)
                        contatore += 1
            print(f"cammini letti: {contatore}")
            return cammini
        else:
            # carica tutto dal pickle
            with open(filepath_pkl, "rb") as f:
                return pickle.load(f)

In [30]:
#paths = load_paths(filepath_pkl="/code/ADSproject/data/cammini.pkl")

In [31]:
#len(paths)

In [32]:
import os
import pickle
import argparse
import bz2
import time

class Graph:

    def __init__(self, directed=False):

        self.adjacency_list = {}  # il grafo sarà un dizionario di dizionari: {nodo: {vicino: peso}} (non usiamo defaultdict per avere più controllo)
        self.directed = directed  # default è False, quindi il grafo è non orientato

## Converte l'identificatore del nodo AS che è una stringa in un intero.

    def _convert_node(self, node):

        try:
            return int(node)
        except (TypeError, ValueError):
            raise ValueError(f"Node {node} is not a valid integer identifier.")

## Definisce come stampare il grafo in modo leggibile

    def __repr__(self):
        graph_str = ""
        for node, neighbors in self.adjacency_list.items():
            graph_str += f" Node {node}: Neighbors and weights {neighbors} \n"
        return graph_str

## Aggiunge un nodo al grafo. Se il nodo esiste già, solleva un'eccezione.

    def add_node(self, node):
        node = self._convert_node(node)

        if node not in self.adjacency_list:
            self.adjacency_list[node] = {}  # aggiunge il nodo con un dizionario (lista di adiacenza e pesi) vuoto
        else:
            raise ValueError(f"Node {node} already exists in the graph.")

## Rimuove un nodo dal grafo. Se il nodo non esiste, solleva un'eccezione.

    def remove_node(self, node):
        node = self._convert_node(node)

        if node not in self.adjacency_list:
            raise ValueError(f"Node {node} does not exist in the graph.")

        for neighbors in self.adjacency_list.values(): ## rimuove il nodo da tutte le liste di adiacenza dei vicini
            neighbors.pop(node, None)

        del self.adjacency_list[node]

## Aggiunge un arco al grafo. Questo arco può essere arbitrario e non proveniente dai cammini BGP (il suo peso sarà None o specificato arbitrariamente dall'utente)

    def add_edge(self, from_node, to_node, weight=None):
        

        #from_node = self._convert_node(from_node) ##ridondanti, lo fa già add_node
        #to_node = self._convert_node(to_node)

        if from_node == to_node:  # elimina i self-loop
            return

        if from_node not in self.adjacency_list:
            self.add_node(from_node)

        if to_node not in self.adjacency_list:
            self.add_node(to_node)

      
        self.adjacency_list[from_node][to_node] = weight

        if not self.directed: ## aggiunge arco in entrambe le direzioni se è undirected
                self.adjacency_list[to_node][from_node] = weight
    
## Rimuove un arco dal grafo. Se i nodi A e B non esistono o se l'arco stesso non esiste (magari i nodi sì ma non sono collegati) lancia un errore.

    def remove_edge(self, from_node, to_node):

        from_node = self._convert_node(from_node)
        to_node = self._convert_node(to_node)

        if from_node not in self.adjacency_list:
            raise ValueError(
                f"Node {from_node} does not exist in the graph."
            )

        if to_node not in self.adjacency_list:
            raise ValueError(
                f"Node {to_node} does not exist in the graph."
            )

        if to_node not in self.adjacency_list[from_node]:
            raise ValueError(
                f"Edge ({from_node}, {to_node}) does not exist in the graph."
            )

        del self.adjacency_list[from_node][to_node]

        if not self.directed:  ## elimina anche l'arco inverso 
            if from_node in self.adjacency_list.get(to_node, {}): ## fa un check se esiste (non dovrebbe servire teoricamente)
                del self.adjacency_list[to_node][from_node]


## Aggiorna la frequenza degli archi

    def update_frequency(self, from_node, to_node):

        from_node = self._convert_node(from_node)
        to_node = self._convert_node(to_node)

        if from_node == to_node:  # elimina i self-loop
            return
        
        ## ho il dubbio che non sia giusto crearli, vediamo

        if from_node not in self.adjacency_list:
            self.add_node(from_node)

        if to_node not in self.adjacency_list:
            self.add_node(to_node)

        #legge la frequenza attuale con .get(...); se l’arco ancora non esiste restituisce 0 come frequenza iniziale e poi aggiunge 1 
        # assegna il nuovo valore a self.adjacency_list[from_node][to_node].

        self.adjacency_list[from_node][to_node] = (                      
            self.adjacency_list[from_node].get(to_node, 0) + 1
        )

        if not self.directed:
            self.adjacency_list[to_node][from_node] = (
                self.adjacency_list[to_node].get(from_node, 0) + 1
            )
    
    def get_neighbors(self, node):
        node = self._convert_node(node)

        if node in self.adjacency_list:
            return self.adjacency_list[node]
        else:
            raise ValueError(f"Node {node} does not exist in the graph.")

    def has_node(self, node):
        node = self._convert_node(node)
        return node in self.adjacency_list

    def has_edge(self, from_node, to_node):
        from_node = self._convert_node(from_node)
        to_node = self._convert_node(to_node)

        if from_node in self.adjacency_list:
            return to_node in self.adjacency_list[from_node]

        return False

    def get_nodes(self):
        return list(self.adjacency_list.keys())

    def get_edges(self):
        edges = []
        seen = set()

        for from_node, neighbors in self.adjacency_list.items():
            for to_node, weight in neighbors.items():

                if not self.directed:
                    arco = (min(from_node, to_node), max(from_node, to_node)) ## prende una unica entry tra (2,3) (3,2)

                    if arco not in seen:
                        seen.add(arco)
                        edges.append((from_node, to_node, weight))

        return edges

    def delete_consecutive_duplicates(self, path): # per esempio path = [10, 10, 20] , conserva solo il primo 10 e 20
        return [
            path[i]
            for i in range(len(path))
            if i == 0 or path[i] != path[i - 1]
        ]

    def add_bgp_path(self, path):
        # converte tutti gli identificatori AS in interi
        path = [self._convert_node(node) for node in path]

        path = self.delete_consecutive_duplicates(path)

        for i in range(len(path) - 1):
            u = path[i]
            v = path[i + 1]

            if u == v:  # elimina i self-loop
                continue

            if not self.has_node(u):
                self.add_node(u)

            if not self.has_node(v):
                self.add_node(v)

            self.update_frequency(u, v)
    

    def largest_connected_component(self):  # Cerchiamo la componente connessa più grande tramite una DFS iterativa.

        visited = set() # nodi già visti

        largest_component = set() # insieme che conterrà i nodi della componente connessa più grande trovata fino a questo momento

        for start_node in self.adjacency_list:

            if start_node in visited:
                continue
                # Saltiamo il resto dell'iterazione e passiamo al nodo successivo se é stato già visto

            component = set() # insieme che conterrà i nodi della componente che stiamo esplorando in questo momento.

            stack = [start_node]
            # Pila utilizzata per eseguire la DFS.
            # Inizialmente contiene solo il nodo di partenza.

            visited.add(start_node)

            while stack:
                # Continuiamo la visita finché ci sono nodi nella pila.

                node = stack.pop()
                # Estraiamo l'ultimo nodo inserito nella pila.
                # Questo comportamento LIFO realizza una DFS.

                component.add(node)
                # Aggiungiamo il nodo alla componente connessa corrente.

                for neighbor in self.adjacency_list[node]:
                    # Scorriamo tutti i vicini del nodo.
                    # Il dizionario interno ha la forma:
                    # {vicino: peso}
                    # Qui vengono considerate solo le chiavi, cioè i vicini.
                    # I pesi non servono per trovare le componenti connesse.

                    if neighbor not in visited:
                        # Consideriamo solo i vicini
                        # che non sono ancora stati visitati.

                        visited.add(neighbor)
                        # Segniamo il vicino come visitato.

                        stack.append(neighbor)
                        # Inseriamo il vicino nella pila,
                        # così verrà esplorato successivamente.

            if len(component) > len(largest_component):
                # Quando la DFS termina, abbiamo trovato
                # un'intera componente connessa.
                # Confrontiamo il suo numero di nodi
                # con quello della componente più grande trovata finora.

                largest_component = component
                # Se la componente corrente è più grande,
                # la salviamo come nuova componente più grande.

        return largest_component
        # Restituiamo l'insieme dei nodi appartenenti
        # alla componente connessa più grande.


    def get_largest_connected_subgraph(self):
        # Troviamo i nodi appartenenti
        # alla componente connessa più grande.

        component_nodes = self.largest_connected_component()

        # Creiamo un nuovo oggetto Graph.
        # Il nuovo grafo mantiene la stessa proprietà del grafo originale:
        # orientato se self.directed è True,
        # non orientato se self.directed è False.

        subgraph = Graph(directed=self.directed)

        for node in component_nodes:
            # Scorriamo tutti i nodi della componente più grande.

            subgraph.add_node(node)
            # Aggiungiamo ogni nodo al nuovo sottografo.
            # In questa fase ogni nodo viene creato
            # con un dizionario dei vicini inizialmente vuoto.

        for from_node in component_nodes:
            # Scorriamo nuovamente tutti i nodi
            # della componente connessa più grande.

            for to_node, weight in self.adjacency_list[from_node].items():
                # Per ogni nodo, scorriamo tutti i suoi vicini
                # e i relativi pesi nel grafo originale.

                if to_node in component_nodes:
                    # Copiamo l'arco soltanto se anche il vicino
                    # appartiene alla componente connessa più grande.

                    subgraph.adjacency_list[from_node][to_node] = weight
                    # Copiamo direttamente l'arco e il suo peso.
                    # Il peso non viene modificato né ricalcolato:
                    # resta uguale a quello presente nel grafo originale.

        return subgraph
        # Restituiamo un nuovo oggetto Graph contenente soltanto
        # la componente connessa più grande.
        # Il grafo originale non viene modificato.

    @staticmethod
    def load_paths(filepath_bz2=None, filepath_pkl=None, max_paths=None):
        
        if filepath_bz2:
            # legge direttamente dal bz2 fermandosi a max_paths
            cammini = []
            contatore = 0
            with bz2.open(filepath_bz2, "rt") as f:
                for riga in f:
                    if contatore >= max_paths:
                        break
                    if riga.startswith("#"):
                        continue
                    parti = riga.strip().split()
                    cammino = []
                    for p in parti[1:]:
                        if "/" in p or "." in p or ":" in p:
                            break
                        nodi = p.split("|")
                        cammino.extend(nodi)
                    if len(cammino) > 1:
                        cammini.append(cammino)
                        contatore += 1
            print(f"cammini letti: {contatore}")
            return cammini
        else:
            # carica tutto dal pickle
            with open(filepath_pkl, "rb") as f:
                return pickle.load(f)

    def build_from_bz2(self, filepath, max_paths=None):

        contatore = 0
        with bz2.open(filepath, "rt") as f:
            for riga in f:
                if max_paths and contatore >= max_paths:
                    break
                if riga.startswith("#"):
                    continue
                parti = riga.strip().split()
                cammino = []
                for p in parti[1:]:
                    if "/" in p or "." in p or ":" in p:
                        break
                    nodi = p.split("|")
                    cammino.extend(nodi)
                if len(cammino) > 1:
                    self.add_bgp_path(cammino)
                    contatore += 1
        print(f"cammini letti: {contatore}")

    def build_from_paths(self, paths):
        for path in paths:
            self.add_bgp_path(path)

    def save_graph(self, filepath):

        with open(filepath, "wb") as f:
            pickle.dump(self, f)

    @staticmethod
    def load_graph(filepath):

        with open(filepath, "rb") as f:
            return pickle.load(f)

In [34]:
import pickle
import time

inizio = time.time()
with open("/code/ADSproject/data/cammini_test.pkl", "rb") as f:
    cammini = pickle.load(f)
fine = time.time()
print(f"tempo caricamento pkl: {fine - inizio:.2f} secondi")

inizio = time.time()
grafo = Graph(directed=False)
grafo.build_from_paths(cammini)  # limita a 1M
fine = time.time()
print(f"tempo costruzione grafo da pkl: {fine - inizio:.2f} secondi")

tempo caricamento pkl: 0.74 secondi
tempo costruzione grafo da pkl: 4.23 secondi


In [36]:
len(grafo.get_edges())


65910

In [37]:
len(grafo.get_nodes())

37020